# 🍃 MongoDB Aggregation Framework with PyMongo
### A Comprehensive Self-Learning Guide

---

## 📋 Learning Objectives

By the end of this notebook, you will be able to:

1. Explain what the Aggregation Framework is and how it differs from `find()`
2. Use common aggregation operators: `$match`, `$group`, `$sort`, `$project`, `$limit`
3. Apply the aggregation framework to real-world use cases
4. Perform cross-collection aggregation using `$lookup`

---

## 📚 Table of Contents

1. [Setup & Sample Data](#setup)
2. [What is the Aggregation Framework?](#what-is-agg)
3. [find() vs. Aggregation Pipeline](#find-vs-agg)
4. [Stage 1: `$match` — Filtering Documents](#match)
5. [Stage 2: `$group` — Grouping & Summarizing](#group)
6. [Stage 3: `$sort` — Ordering Results](#sort)
7. [Stage 4: `$project` — Reshaping Documents](#project)
8. [Stage 5: `$limit` — Restricting Output](#limit)
9. [Combining Multiple Stages Together](#combining)
10. [Real-World Use Cases](#usecases)
11. [Cross-Collection Aggregation with `$lookup`](#lookup)
12. [Summary & Quick Reference](#summary)


---
<a id='setup'></a>
## 🔧 Section 1: Setup & Sample Data

Before we begin, let's install the required library and prepare a **local in-memory simulation** of MongoDB data using Python dictionaries. This way you can run this notebook **without needing a running MongoDB server**.

> **📌 Note for Instructors/Students with MongoDB installed:** If you have MongoDB running locally or on Atlas, replace the simulation section with a real `pymongo.MongoClient()` connection. All aggregation pipeline syntax is identical.

### 1.1 Install PyMongo

In [ ]:
# Install pymongo if not already installed
!pip install pymongo --quiet
print("✅ pymongo is ready!")

### 1.2 Connect to MongoDB

We will use `mongomock` for an in-memory MongoDB simulation, which is perfect for learning without needing a server.

If you have a real MongoDB running, simply replace `mongomock.MongoClient()` with `pymongo.MongoClient('mongodb://localhost:27017/')`.

In [ ]:
# Install mongomock for in-memory simulation
!pip install mongomock --quiet

import mongomock
import pymongo
import pprint
from datetime import datetime

# ─────────────────────────────────────────────────────────
# OPTION A: In-memory simulation (no MongoDB server needed)
# ─────────────────────────────────────────────────────────
client = mongomock.MongoClient()

# ─────────────────────────────────────────────────────────
# OPTION B: Real MongoDB (uncomment the line below)
# ─────────────────────────────────────────────────────────
# client = pymongo.MongoClient('mongodb://localhost:27017/')

# Select database
db = client['university_db']

print("✅ Connected to MongoDB!")
print(f"   Database: university_db")

### 1.3 Load Sample Data

We will use a **University dataset** with three collections:

| Collection | Description |
|---|---|
| `students` | Student profiles with grades and departments |
| `courses` | Course catalog with credits and department |
| `enrollments` | Records of which student is taking which course |

This simulates a realistic academic data environment.

In [ ]:
# ──────────────────────────────────
# Drop existing collections (clean start)
# ──────────────────────────────────
db.students.drop()
db.courses.drop()
db.enrollments.drop()

# ──────────────────────────────────
# Students Collection
# ──────────────────────────────────
students_data = [
    {"student_id": "S001", "name": "Alice Tan",    "department": "Computer Science", "year": 2, "gpa": 3.8, "status": "active"},
    {"student_id": "S002", "name": "Bob Lim",      "department": "Data Science",      "year": 3, "gpa": 3.2, "status": "active"},
    {"student_id": "S003", "name": "Carol Wong",   "department": "Computer Science", "year": 1, "gpa": 3.5, "status": "active"},
    {"student_id": "S004", "name": "David Ong",    "department": "Data Science",      "year": 2, "gpa": 2.9, "status": "inactive"},
    {"student_id": "S005", "name": "Eve Hassan",   "department": "Computer Science", "year": 4, "gpa": 3.9, "status": "active"},
    {"student_id": "S006", "name": "Frank Ali",    "department": "Software Engineering", "year": 2, "gpa": 3.1, "status": "active"},
    {"student_id": "S007", "name": "Grace Ng",     "department": "Data Science",      "year": 1, "gpa": 3.7, "status": "active"},
    {"student_id": "S008", "name": "Henry Yap",    "department": "Software Engineering", "year": 3, "gpa": 2.7, "status": "active"},
    {"student_id": "S009", "name": "Irene Soo",    "department": "Computer Science", "year": 2, "gpa": 3.6, "status": "active"},
    {"student_id": "S010", "name": "James Foo",    "department": "Software Engineering", "year": 4, "gpa": 3.0, "status": "inactive"},
]

# ──────────────────────────────────
# Courses Collection
# ──────────────────────────────────
courses_data = [
    {"course_id": "CS101", "title": "Introduction to Programming",   "department": "Computer Science",    "credits": 3, "level": "undergraduate"},
    {"course_id": "CS202", "title": "Data Structures",               "department": "Computer Science",    "credits": 4, "level": "undergraduate"},
    {"course_id": "DS101", "title": "Introduction to Data Science",  "department": "Data Science",        "credits": 3, "level": "undergraduate"},
    {"course_id": "DS301", "title": "Machine Learning",              "department": "Data Science",        "credits": 4, "level": "postgraduate"},
    {"course_id": "SE101", "title": "Software Engineering Basics",   "department": "Software Engineering","credits": 3, "level": "undergraduate"},
    {"course_id": "CS303", "title": "NoSQL Databases",               "department": "Computer Science",    "credits": 3, "level": "postgraduate"},
]

# ──────────────────────────────────
# Enrollments Collection
# ──────────────────────────────────
enrollments_data = [
    {"student_id": "S001", "course_id": "CS101", "semester": "2024-1", "grade": "A",  "score": 90},
    {"student_id": "S001", "course_id": "CS202", "semester": "2024-1", "grade": "A-", "score": 87},
    {"student_id": "S002", "course_id": "DS101", "semester": "2024-1", "grade": "B+", "score": 78},
    {"student_id": "S002", "course_id": "DS301", "semester": "2024-1", "grade": "B",  "score": 75},
    {"student_id": "S003", "course_id": "CS101", "semester": "2024-1", "grade": "A",  "score": 92},
    {"student_id": "S003", "course_id": "CS303", "semester": "2024-2", "grade": "A-", "score": 88},
    {"student_id": "S004", "course_id": "DS101", "semester": "2024-1", "grade": "C+", "score": 65},
    {"student_id": "S005", "course_id": "CS202", "semester": "2024-1", "grade": "A",  "score": 95},
    {"student_id": "S005", "course_id": "CS303", "semester": "2024-2", "grade": "A",  "score": 97},
    {"student_id": "S006", "course_id": "SE101", "semester": "2024-1", "grade": "B",  "score": 74},
    {"student_id": "S007", "course_id": "DS101", "semester": "2024-1", "grade": "A-", "score": 86},
    {"student_id": "S007", "course_id": "DS301", "semester": "2024-2", "grade": "A",  "score": 93},
    {"student_id": "S008", "course_id": "SE101", "semester": "2024-1", "grade": "C",  "score": 62},
    {"student_id": "S009", "course_id": "CS101", "semester": "2024-1", "grade": "A-", "score": 89},
    {"student_id": "S009", "course_id": "CS303", "semester": "2024-2", "grade": "B+", "score": 79},
    {"student_id": "S010", "course_id": "SE101", "semester": "2024-1", "grade": "B-", "score": 71},
]

# Insert into collections
db.students.insert_many(students_data)
db.courses.insert_many(courses_data)
db.enrollments.insert_many(enrollments_data)

print("✅ Sample data inserted successfully!")
print(f"   students    : {db.students.count_documents({})} documents")
print(f"   courses     : {db.courses.count_documents({})} documents")
print(f"   enrollments : {db.enrollments.count_documents({})} documents")

### 1.4 Helper Function: Pretty Print Results

We will create a small helper to display results in a clean, readable format.

In [ ]:
def show_results(cursor, title="Results", max_docs=10):
    """
    Display aggregation or find() results in a readable format.
    
    Args:
        cursor   : MongoDB cursor or list of documents
        title    : Label to display above results
        max_docs : Maximum number of documents to display
    """
    docs = list(cursor)
    print(f"\n{'='*55}")
    print(f"  📊 {title}")
    print(f"{'='*55}")
    
    if not docs:
        print("  (No results found)")
    else:
        for i, doc in enumerate(docs[:max_docs]):
            # Remove the _id field for cleaner display
            doc.pop('_id', None)
            print(f"  [{i+1}] ", end="")
            pprint.pprint(doc, indent=6, width=80)
        
        if len(docs) > max_docs:
            print(f"  ... and {len(docs) - max_docs} more document(s)")
    
    print(f"  Total: {len(docs)} document(s) returned")
    print(f"{'='*55}\n")

print("✅ Helper function 'show_results()' is ready!")

---
<a id='what-is-agg'></a>
## 🧠 Section 2: What is the Aggregation Framework?

### 2.1 The Concept

The **MongoDB Aggregation Framework** is a powerful data processing engine that allows you to **transform, group, filter, and compute** data across many documents — directly inside the database, without pulling all data into your application.

Think of it like an **assembly line** in a factory:

```
Raw Documents
     ↓
 [ Stage 1: Filter ]   ← $match
     ↓
 [ Stage 2: Group  ]   ← $group
     ↓
 [ Stage 3: Sort   ]   ← $sort
     ↓
 [ Stage 4: Shape  ]   ← $project
     ↓
 Final Results ✅
```

Each **stage** takes documents as input and passes them (possibly transformed) to the next stage. This chain of stages is called the **Aggregation Pipeline**.

### 2.2 Why Do We Need It?

Imagine you want to answer: _"What is the average GPA per department for active students?"_

Without aggregation, you would:
1. Fetch ALL students with `find()`
2. Loop through them in Python
3. Manually group and compute averages

With aggregation, MongoDB does all the heavy lifting **server-side**, which is:
- ✅ **Faster** — processed where the data lives
- ✅ **More efficient** — reduces network transfer
- ✅ **Cleaner code** — no complex Python loops

---
<a id='find-vs-agg'></a>
## ⚖️ Section 3: `find()` vs. Aggregation Pipeline

### 3.1 Key Differences

| Feature | `find()` | Aggregation Pipeline |
|---|---|---|
| **Purpose** | Retrieve documents as-is | Transform, compute, reshape |
| **Filtering** | Yes (`query` argument) | Yes (`$match` stage) |
| **Grouping** | ❌ Not supported | ✅ `$group` stage |
| **Computed Fields** | Limited | ✅ Full `$project` support |
| **Cross-collection joins** | ❌ Not supported | ✅ `$lookup` stage |
| **Output shape** | Same as input documents | Fully customizable |
| **Method** | `collection.find()` | `collection.aggregate([...])` |

### 3.2 Side-by-Side Example

**Goal:** Get all active students from the Computer Science department.

In [ ]:
# ─────────────────────────────────────────
# APPROACH 1: Using find()
# ─────────────────────────────────────────
# Syntax: db.collection.find( {filter}, {projection} )

find_results = db.students.find(
    {
        "department": "Computer Science",
        "status": "active"
    },
    {
        "_id": 0,
        "name": 1,
        "department": 1,
        "gpa": 1
    }
)

show_results(find_results, title="find() — Active CS Students")

In [ ]:
# ─────────────────────────────────────────
# APPROACH 2: Using Aggregation Pipeline
# ─────────────────────────────────────────
# Syntax: db.collection.aggregate( [ stage1, stage2, ... ] )

agg_results = db.students.aggregate([
    # Stage 1: Filter documents
    {
        "$match": {
            "department": "Computer Science",
            "status": "active"
        }
    },
    # Stage 2: Reshape the output
    {
        "$project": {
            "_id": 0,
            "name": 1,
            "department": 1,
            "gpa": 1
        }
    }
])

show_results(agg_results, title="aggregate() — Active CS Students")

> **💡 Observation:** For simple retrieval, `find()` is shorter and perfectly fine. The aggregation pipeline shines when you need **grouping, computation, or multi-stage transformations** — things `find()` simply cannot do.

---
<a id='match'></a>
## 🔍 Section 4: The `$match` Stage — Filtering Documents

### 4.1 What is `$match`?

`$match` works like a filter gate — it only passes documents that satisfy the given conditions to the next stage. It uses the **same query syntax as `find()`**, so it should feel familiar.

**Syntax:**
```python
{ "$match": { <field>: <condition> } }
```

> **⚡ Performance Tip:** Always place `$match` as **early as possible** in the pipeline. This reduces the number of documents flowing into later stages, making the pipeline faster.

### 4.2 Example 1 — Simple Equality Filter

**Goal:** Find all students in the Data Science department.

In [ ]:
pipeline = [
    {
        "$match": {
            "department": "Data Science"   # exact match
        }
    }
]

results = db.students.aggregate(pipeline)
show_results(results, title="$match — Data Science Students")

### 4.3 Example 2 — Compound Conditions

**Goal:** Find active students with GPA greater than or equal to 3.5.

In [ ]:
pipeline = [
    {
        "$match": {
            "status": "active",        # must be active
            "gpa": {"$gte": 3.5}       # AND gpa >= 3.5
        }
    }
]

results = db.students.aggregate(pipeline)
show_results(results, title="$match — Active Students with GPA ≥ 3.5")

### 4.4 Example 3 — Using `$or` Logical Operator

**Goal:** Find students who are either in Year 1 OR have a GPA above 3.8.

In [ ]:
pipeline = [
    {
        "$match": {
            "$or": [
                {"year": 1},              # Year 1 students
                {"gpa": {"$gt": 3.8}}    # OR GPA above 3.8
            ]
        }
    }
]

results = db.students.aggregate(pipeline)
show_results(results, title="$match with $or — Year 1 OR GPA > 3.8")

---
<a id='group'></a>
## 📦 Section 5: The `$group` Stage — Grouping & Summarizing

### 5.1 What is `$group`?

`$group` is one of the most powerful aggregation stages. It groups documents that share a common value and allows you to **compute aggregate values** (sum, average, count, min, max) for each group.

**Syntax:**
```python
{
    "$group": {
        "_id": "$fieldName",         # Group by this field
        "newField": { "$operator": "$anotherField" }  # Computed value
    }
}
```

**Common Accumulator Operators:**

| Operator | Description | Example Use |
|---|---|---|
| `$sum` | Add up values | Total scores, count with `1` |
| `$avg` | Compute average | Average GPA per department |
| `$min` | Find minimum | Lowest score in a group |
| `$max` | Find maximum | Highest GPA in a group |
| `$count` | Count documents | Number of students per dept |
| `$push` | Build an array | List of student names per dept |
| `$first` | First value in group | — |
| `$last` | Last value in group | — |

### 5.2 Example 1 — Count Students per Department

In [ ]:
pipeline = [
    {
        "$group": {
            "_id": "$department",          # Group by department field
            "total_students": {"$sum": 1}  # Count each document as 1
        }
    }
]

results = db.students.aggregate(pipeline)
show_results(results, title="$group — Student Count per Department")

### 5.3 Example 2 — Average GPA per Department

In [ ]:
pipeline = [
    {
        "$group": {
            "_id": "$department",
            "average_gpa":  {"$avg": "$gpa"},    # Average of gpa field
            "highest_gpa": {"$max": "$gpa"},     # Maximum gpa in group
            "lowest_gpa":  {"$min": "$gpa"}      # Minimum gpa in group
        }
    }
]

results = db.students.aggregate(pipeline)
show_results(results, title="$group — GPA Statistics per Department")

### 5.4 Example 3 — Grouping with `$push` to Build Arrays

**Goal:** For each department, collect the names of all students into a list.

In [ ]:
pipeline = [
    {
        "$group": {
            "_id": "$department",
            "student_names": {"$push": "$name"}   # Build an array of names
        }
    }
]

results = db.students.aggregate(pipeline)
show_results(results, title="$group with $push — Students per Department")

### 5.5 Example 4 — Group by Multiple Fields

**Goal:** Count students grouped by **both** department AND year.

In [ ]:
pipeline = [
    {
        "$group": {
            # Group by a COMPOUND key (multiple fields)
            "_id": {
                "department": "$department",
                "year": "$year"
            },
            "count": {"$sum": 1}
        }
    }
]

results = db.students.aggregate(pipeline)
show_results(results, title="$group — Count by Department AND Year")

---
<a id='sort'></a>
## 📊 Section 6: The `$sort` Stage — Ordering Results

### 6.1 What is `$sort`?

`$sort` orders the documents coming through the pipeline. It can sort on any field — including computed fields created by earlier stages.

**Syntax:**
```python
{ "$sort": { "fieldName": 1 } }   # 1 = Ascending (A→Z, small→big)
{ "$sort": { "fieldName": -1 } }  # -1 = Descending (Z→A, big→small)
```

### 6.2 Example 1 — Sort Students by GPA (Descending)

In [ ]:
pipeline = [
    {
        "$sort": {
            "gpa": -1    # -1 = Descending (highest first)
        }
    }
]

results = db.students.aggregate(pipeline)
show_results(results, title="$sort — Students by GPA (Highest First)")

### 6.3 Example 2 — Sort on Computed Fields (After `$group`)

**Goal:** Group students by department, compute average GPA, then sort departments from highest to lowest average GPA.

In [ ]:
pipeline = [
    # Stage 1: Group and compute average GPA
    {
        "$group": {
            "_id": "$department",
            "avg_gpa": {"$avg": "$gpa"},
            "student_count": {"$sum": 1}
        }
    },
    # Stage 2: Sort by the newly computed field 'avg_gpa'
    {
        "$sort": {
            "avg_gpa": -1    # Highest average GPA first
        }
    }
]

results = db.students.aggregate(pipeline)
show_results(results, title="$group + $sort — Departments Ranked by Avg GPA")

### 6.4 Example 3 — Multi-Field Sort

**Goal:** Sort students first by department name (ascending), then within each department by GPA (descending).

In [ ]:
pipeline = [
    {
        "$sort": {
            "department": 1,   # Primary sort: department A→Z
            "gpa": -1           # Secondary sort: GPA high→low
        }
    }
]

results = db.students.aggregate(pipeline)
show_results(results, title="$sort — By Department (A→Z), then GPA (High→Low)")

---
<a id='project'></a>
## 🎨 Section 7: The `$project` Stage — Reshaping Documents

### 7.1 What is `$project`?

`$project` controls which fields appear in the output documents. It can:
- **Include** specific fields (`1`)
- **Exclude** specific fields (`0`)
- **Create computed/new fields** using expressions
- **Rename fields**

**Syntax:**
```python
{
    "$project": {
        "fieldToInclude": 1,
        "fieldToExclude": 0,
        "newFieldName": "$existingField",  # rename
        "computedField": { "$multiply": ["$a", "$b"] }  # expression
    }
}
```

### 7.2 Example 1 — Include & Exclude Fields

In [ ]:
pipeline = [
    {
        "$project": {
            "_id": 0,           # Exclude _id
            "name": 1,          # Include name
            "department": 1,    # Include department
            "gpa": 1            # Include gpa
            # 'year', 'status', 'student_id' are excluded automatically
        }
    }
]

results = db.students.aggregate(pipeline)
show_results(results, title="$project — Show Only Name, Department, GPA")

### 7.3 Example 2 — Rename a Field

In [ ]:
pipeline = [
    {
        "$project": {
            "_id": 0,
            # Rename fields
            "full_name": "$name",          # 'name' → 'full_name'
            "dept": "$department",         # 'department' → 'dept'
            "grade_point_avg": "$gpa"      # 'gpa' → 'grade_point_avg'
        }
    }
]

results = db.students.aggregate(pipeline)
show_results(results, title="$project — Rename Fields")

### 7.4 Example 3 — Create Computed Fields

**Goal:** Create a field `gpa_percentage` by multiplying GPA by 25 (scaling 4.0 → 100%).

In [ ]:
pipeline = [
    {
        "$project": {
            "_id": 0,
            "name": 1,
            "gpa": 1,
            # Compute a new field: gpa_percentage = gpa * 25
            "gpa_percentage": {
                "$multiply": ["$gpa", 25]
            },
            # Compute academic_level from 'year' field
            "academic_level": {
                "$cond": {
                    "if":   {"$lte": ["$year", 2]},
                    "then": "Junior",
                    "else": "Senior"
                }
            }
        }
    }
]

results = db.students.aggregate(pipeline)
show_results(results, title="$project — Computed Fields")

---
<a id='limit'></a>
## 🔢 Section 8: The `$limit` Stage — Restricting Output

### 8.1 What is `$limit`?

`$limit` restricts the number of documents that pass to the next stage (or appear in the final output). It is commonly used to:
- Return only the **top N results** (e.g., Top 3 students)
- Implement **pagination** together with `$skip`

**Syntax:**
```python
{ "$limit": N }   # N is a positive integer
```

> **⚠️ Important:** Always use `$sort` BEFORE `$limit` if you want meaningful top-N results. Without sorting first, MongoDB returns whatever N documents it encounters first.

### 8.2 Example 1 — Top 3 Students by GPA

In [ ]:
pipeline = [
    # Step 1: Sort by GPA (highest first)
    {
        "$sort": {"gpa": -1}
    },
    # Step 2: Take only the first 3
    {
        "$limit": 3
    },
    # Step 3: Clean up the output
    {
        "$project": {
            "_id": 0,
            "name": 1,
            "department": 1,
            "gpa": 1
        }
    }
]

results = db.students.aggregate(pipeline)
show_results(results, title="$sort + $limit — Top 3 Students by GPA")

### 8.3 Example 2 — Pagination using `$skip` and `$limit`

**Goal:** Simulate a paginated list — show students ranked 4 to 6 (Page 2 if page size = 3).

In [ ]:
page_number = 2    # Which page do you want?
page_size   = 3    # How many per page?

pipeline = [
    {"$sort": {"gpa": -1}},                        # Sort
    {"$skip": (page_number - 1) * page_size},       # Skip previous pages
    {"$limit": page_size},                          # Take current page
    {"$project": {"_id": 0, "name": 1, "gpa": 1, "department": 1}}
]

results = db.students.aggregate(pipeline)
show_results(results, title=f"Pagination — Page {page_number} (students ranked 4-6)")

---
<a id='combining'></a>
## 🔗 Section 9: Combining Multiple Stages Together

The real power of the aggregation framework comes from **chaining stages together**. Let's solve progressively complex problems.

### 9.1 Example — Full Pipeline: Filter → Group → Sort → Limit → Project

**Goal:** Find the top 2 departments with the **highest average GPA**, but only count **active** students.

In [ ]:
pipeline = [
    # Stage 1: $match — Only active students
    {
        "$match": {
            "status": "active"
        }
    },
    
    # Stage 2: $group — Group by department, compute stats
    {
        "$group": {
            "_id": "$department",
            "avg_gpa":       {"$avg": "$gpa"},
            "total_students":{"$sum": 1},
            "top_student":   {"$max": "$gpa"}
        }
    },
    
    # Stage 3: $sort — Highest avg GPA first
    {
        "$sort": {
            "avg_gpa": -1
        }
    },
    
    # Stage 4: $limit — Top 2 departments only
    {
        "$limit": 2
    },
    
    # Stage 5: $project — Rename and round values
    {
        "$project": {
            "_id": 0,
            "department": "$_id",
            "average_gpa": {"$round": ["$avg_gpa", 2]},
            "total_students": 1,
            "highest_gpa_in_dept": "$top_student"
        }
    }
]

results = db.students.aggregate(pipeline)
show_results(results, title="Full Pipeline — Top 2 Departments by Avg GPA (Active Only)")

---
<a id='usecases'></a>
## 🌍 Section 10: Real-World Use Cases

Now let's apply the aggregation framework to realistic scenarios you might encounter in industry or research.

---

### Use Case 1 — 📈 Grade Distribution Analysis

**Scenario:** An academic administrator wants to see the grade distribution across all courses — how many A's, B's, C's were given?

In [ ]:
pipeline = [
    # Group by grade, count how many students received each grade
    {
        "$group": {
            "_id": "$grade",
            "student_count":  {"$sum": 1},
            "average_score": {"$avg": "$score"}
        }
    },
    # Sort alphabetically by grade
    {
        "$sort": {"_id": 1}
    },
    {
        "$project": {
            "_id": 0,
            "grade": "$_id",
            "student_count": 1,
            "average_score": {"$round": ["$average_score", 1]}
        }
    }
]

results = db.enrollments.aggregate(pipeline)
show_results(results, title="Use Case 1 — Grade Distribution")

---

### Use Case 2 — 🏆 Top Performing Students per Semester

**Scenario:** Find the top 3 highest-scoring students for each semester.

In [ ]:
pipeline = [
    # Group by semester AND student, compute their average score
    {
        "$group": {
            "_id": {
                "semester":   "$semester",
                "student_id": "$student_id"
            },
            "avg_score": {"$avg": "$score"}
        }
    },
    # Sort by semester ascending, then score descending
    {
        "$sort": {
            "_id.semester": 1,
            "avg_score": -1
        }
    },
    # Group by semester, collect top students using $push, then slice to top 3
    {
        "$group": {
            "_id": "$_id.semester",
            "top_students": {
                "$push": {
                    "student_id": "$_id.student_id",
                    "avg_score": {"$round": ["$avg_score", 1]}
                }
            }
        }
    },
    # Keep only top 3 per semester using $slice
    {
        "$project": {
            "_id": 0,
            "semester": "$_id",
            "top_3_students": {"$slice": ["$top_students", 3]}
        }
    },
    {"$sort": {"semester": 1}}
]

results = db.enrollments.aggregate(pipeline)
show_results(results, title="Use Case 2 — Top 3 Students per Semester")

---

### Use Case 3 — 📉 Identify At-Risk Students

**Scenario:** A student support office wants to identify active students with a GPA below 3.0, sorted by GPA (most urgent first).

In [ ]:
pipeline = [
    # Filter: Active students with GPA below threshold
    {
        "$match": {
            "status": "active",
            "gpa": {"$lt": 3.2}   # Below 3.2 threshold
        }
    },
    # Sort: Lowest GPA first (most urgent)
    {
        "$sort": {"gpa": 1}
    },
    # Project: Only show relevant counseling fields
    {
        "$project": {
            "_id": 0,
            "student_id": 1,
            "name": 1,
            "department": 1,
            "year": 1,
            "gpa": 1,
            "alert_level": {
                "$cond": {
                    "if":   {"$lt": ["$gpa", 3.0]},
                    "then": "🔴 HIGH",
                    "else": "🟡 MEDIUM"
                }
            }
        }
    }
]

results = db.students.aggregate(pipeline)
show_results(results, title="Use Case 3 — At-Risk Student Alert Report")

---

### Use Case 4 — 📊 Course Popularity by Enrollment Count

**Scenario:** Which courses have the most enrollments? Rank them from most to least popular.

In [ ]:
pipeline = [
    # Group by course and count enrollments
    {
        "$group": {
            "_id": "$course_id",
            "enrollments":  {"$sum": 1},
            "avg_score": {"$avg": "$score"}
        }
    },
    # Sort: most enrolled first
    {
        "$sort": {"enrollments": -1}
    },
    # Rename and clean
    {
        "$project": {
            "_id": 0,
            "course_id": "$_id",
            "enrollments": 1,
            "avg_score": {"$round": ["$avg_score", 1]}
        }
    }
]

results = db.enrollments.aggregate(pipeline)
show_results(results, title="Use Case 4 — Course Popularity Ranking")

---
<a id='lookup'></a>
## 🔗 Section 11: Cross-Collection Aggregation with `$lookup`

### 11.1 Can We Aggregate Across Different Collections?

**Yes! MongoDB supports cross-collection aggregation using the `$lookup` stage.**

`$lookup` performs a **left outer join** — it fetches matching documents from another collection and embeds them as an array field in the current document. This is MongoDB's equivalent of SQL's `JOIN`.

**Syntax:**
```python
{
    "$lookup": {
        "from":         "otherCollection",   # The collection to JOIN
        "localField":   "fieldInCurrentDoc", # Matching field in current collection
        "foreignField": "fieldInOtherDoc",   # Matching field in other collection
        "as":           "resultArrayName"    # Name of the joined array
    }
}
```

**Visualized:**
```
enrollments collection          courses collection
──────────────────────          ──────────────────
{ course_id: "CS101",  }  ───►  { course_id: "CS101",
  student_id: "S001",              title: "Intro to Programming",
  score: 90 }                      credits: 3 }

                    ↓ After $lookup ↓

{ course_id: "CS101",
  student_id: "S001",
  score: 90,
  course_info: [{ title: "Intro to Programming", credits: 3, ... }] }
```

### 11.2 Example 1 — Join Enrollments with Course Details

**Goal:** For each enrollment record, attach the course's title, department, and credit information.

In [ ]:
pipeline = [
    # Stage 1: $lookup — Join enrollments with courses
    {
        "$lookup": {
            "from":         "courses",    # Join with 'courses' collection
            "localField":   "course_id",  # Field in enrollments
            "foreignField": "course_id",  # Field in courses
            "as":           "course_info" # Result stored in this array
        }
    },
    # Stage 2: $unwind — Flatten the array (since each enrollment matches 1 course)
    {
        "$unwind": "$course_info"
    },
    # Stage 3: $project — Select and rename meaningful fields
    {
        "$project": {
            "_id": 0,
            "student_id": 1,
            "course_title":  "$course_info.title",
            "department":    "$course_info.department",
            "credits":       "$course_info.credits",
            "semester": 1,
            "grade": 1,
            "score": 1
        }
    },
    # Stage 4: Sort by student_id
    {"$sort": {"student_id": 1}}
]

results = db.enrollments.aggregate(pipeline)
show_results(results, title="$lookup — Enrollments with Course Details", max_docs=6)

### 11.3 Example 2 — Three-Way Join: Enrollments + Students + Courses

**Goal:** Build a complete report that shows student name, course title, score, and semester — joining all three collections.

In [ ]:
pipeline = [
    # Step 1: Join enrollments → students
    {
        "$lookup": {
            "from":         "students",
            "localField":   "student_id",
            "foreignField": "student_id",
            "as":           "student_info"
        }
    },
    {"$unwind": "$student_info"},  # Flatten
    
    # Step 2: Join enrollments → courses
    {
        "$lookup": {
            "from":         "courses",
            "localField":   "course_id",
            "foreignField": "course_id",
            "as":           "course_info"
        }
    },
    {"$unwind": "$course_info"},   # Flatten
    
    # Step 3: Shape the final output
    {
        "$project": {
            "_id": 0,
            "student_name":   "$student_info.name",
            "course_title":   "$course_info.title",
            "semester": 1,
            "grade": 1,
            "score": 1,
            "credits_earned": "$course_info.credits"
        }
    },
    
    # Step 4: Sort by score descending
    {"$sort": {"score": -1}}
]

results = db.enrollments.aggregate(pipeline)
show_results(results, title="Three-Way Join — Full Academic Report", max_docs=8)

### 11.4 Example 3 — Aggregate After Join: Credits per Student

**Goal:** Calculate the **total credits** each student has earned (via their enrollments), then rank them.

In [ ]:
pipeline = [
    # Step 1: Join enrollments → courses (to get credit info)
    {
        "$lookup": {
            "from":         "courses",
            "localField":   "course_id",
            "foreignField": "course_id",
            "as":           "course_info"
        }
    },
    {"$unwind": "$course_info"},
    
    # Step 2: Join enrollments → students (to get student names)
    {
        "$lookup": {
            "from":         "students",
            "localField":   "student_id",
            "foreignField": "student_id",
            "as":           "student_info"
        }
    },
    {"$unwind": "$student_info"},
    
    # Step 3: Group by student, SUM their credits
    {
        "$group": {
            "_id": "$student_id",
            "student_name":   {"$first": "$student_info.name"},
            "department":     {"$first": "$student_info.department"},
            "total_credits":  {"$sum": "$course_info.credits"},
            "courses_taken":  {"$sum": 1}
        }
    },
    
    # Step 4: Sort by total credits (highest first)
    {"$sort": {"total_credits": -1}},
    
    # Step 5: Final projection
    {
        "$project": {
            "_id": 0,
            "student_name": 1,
            "department": 1,
            "total_credits": 1,
            "courses_taken": 1
        }
    }
]

results = db.enrollments.aggregate(pipeline)
show_results(results, title="Credits Earned per Student (Ranked)")

---
<a id='summary'></a>
## 📝 Section 12: Summary & Quick Reference

### 12.1 Aggregation vs. find() — When to Use Which?

| Task | Use `find()` | Use `aggregate()` |
|---|---|---|
| Fetch documents as-is | ✅ | Can, but overkill |
| Filter documents | ✅ | ✅ (via `$match`) |
| Count or sum values | ❌ | ✅ (via `$group`) |
| Compute new fields | ❌ | ✅ (via `$project`) |
| Sort results | ✅ (`.sort()`) | ✅ (via `$sort`) |
| Join collections | ❌ | ✅ (via `$lookup`) |
| Top-N results | ✅ (`.limit()`) | ✅ (via `$limit`) |

In [ ]:
# ─────────────────────────────────────────────────────────────────
# QUICK REFERENCE: All operators in one example
# ─────────────────────────────────────────────────────────────────

print("""
╔══════════════════════════════════════════════════════════════╗
║         MongoDB Aggregation — Stage Quick Reference          ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  Stage      Purpose                    Key Args              ║
║  ─────────  ─────────────────────────  ──────────────────    ║
║  $match     Filter documents           Same as find()        ║
║  $group     Group + compute values     _id, accumulators     ║
║  $sort      Order results              field: 1 or -1        ║
║  $project   Include/exclude/compute    field: 1/0/expr       ║
║  $limit     Restrict output count      integer N             ║
║  $skip      Skip first N documents     integer N             ║
║  $lookup    Join another collection    from, localField...   ║
║  $unwind    Flatten array field        "$arrayField"         ║
║  $addFields Add/overwrite fields       field: expression     ║
║  $count     Count documents            string (field name)   ║
║                                                              ║
║  Accumulators ($group only):                                 ║
║  $sum, $avg, $min, $max, $push, $first, $last, $count        ║
║                                                              ║
║  Expressions ($project, $match, $group):                     ║
║  $multiply, $divide, $add, $subtract                         ║
║  $concat, $toUpper, $toLower                                 ║
║  $cond, $ifNull, $switch                                     ║
║  $round, $floor, $ceil, $abs                                 ║
║  $slice, $size, $arrayElemAt                                 ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")

### 12.2 The Golden Rules of Aggregation Pipelines

1. **`$match` goes first** — filter early to reduce data flowing through.
2. **`$sort` before `$limit`** — always sort before limiting to get meaningful top-N results.
3. **`$unwind` after `$lookup`** — `$lookup` produces arrays; `$unwind` flattens them for further processing.
4. **`$project` goes last** — use it to clean up and shape the final output.
5. **Use `$group` for computations** — sum, count, average are only possible here.

### 12.3 Final Challenge — Try It Yourself! 🎯

Write an aggregation pipeline that answers:

> _"For each department, find the student who has enrolled in the most courses and list their name, department, and enrollment count."_

**Hint:** You'll need `$lookup`, `$group`, `$sort`, and `$project`.

In [ ]:
# ─────────────────────────────────────────
# YOUR ANSWER HERE
# ─────────────────────────────────────────

my_pipeline = [
    # TODO: Step 1 — Join enrollments with students
    
    # TODO: Step 2 — Group by student, count courses
    
    # TODO: Step 3 — Sort within each department
    
    # TODO: Step 4 — Group by department, pick the top student
    
    # TODO: Step 5 — Project clean output
]

# results = db.enrollments.aggregate(my_pipeline)
# show_results(results, title="Challenge — Most Active Student per Department")

print("📝 Fill in the pipeline above and uncomment the last two lines to test!")

In [ ]:
# ─────────────────────────────────────────
# SAMPLE SOLUTION (Expand after attempting)
# ─────────────────────────────────────────

solution_pipeline = [
    # Step 1: Join enrollments → students to get dept and name
    {
        "$lookup": {
            "from": "students",
            "localField": "student_id",
            "foreignField": "student_id",
            "as": "student_info"
        }
    },
    {"$unwind": "$student_info"},
    
    # Step 2: Group by student, count courses
    {
        "$group": {
            "_id": "$student_id",
            "name":       {"$first": "$student_info.name"},
            "department": {"$first": "$student_info.department"},
            "course_count": {"$sum": 1}
        }
    },
    
    # Step 3: Sort by department, then course_count descending
    {
        "$sort": {
            "department": 1,
            "course_count": -1
        }
    },
    
    # Step 4: Group by department, take the first student (highest count)
    {
        "$group": {
            "_id": "$department",
            "top_student":  {"$first": "$name"},
            "course_count": {"$first": "$course_count"}
        }
    },
    
    # Step 5: Clean output
    {
        "$project": {
            "_id": 0,
            "department": "$_id",
            "most_active_student": "$top_student",
            "courses_enrolled": "$course_count"
        }
    },
    {"$sort": {"department": 1}}
]

results = db.enrollments.aggregate(solution_pipeline)
show_results(results, title="✅ Solution — Most Active Student per Department")

---

## 🎉 Congratulations!

You have completed the **MongoDB Aggregation Framework** self-learning guide. Here's what you've learned:

| Topic | Status |
|---|---|
| What is the Aggregation Framework | ✅ |
| `find()` vs `aggregate()` differences | ✅ |
| `$match` — filtering documents | ✅ |
| `$group` — grouping and computing | ✅ |
| `$sort` — ordering results | ✅ |
| `$project` — reshaping documents | ✅ |
| `$limit` & `$skip` — pagination | ✅ |
| Combining multiple stages | ✅ |
| Real-world use cases | ✅ |
| Cross-collection joins with `$lookup` | ✅ |

---

### 📚 Further Reading
- [MongoDB Official Aggregation Documentation](https://www.mongodb.com/docs/manual/aggregation/)
- [PyMongo Documentation](https://pymongo.readthedocs.io/)
- [MongoDB Aggregation Pipeline Operators](https://www.mongodb.com/docs/manual/reference/operator/aggregation/)

---
*Prepared for NoSQL Databases Course | CLO1 · CLO2 · CLO4*